# Gemma3-4B — 移行版
環境依存の認証・パス・インストールを調整した実験ログです。全セル一括実行用ではありません。
元のセル順・実験条件は維持しています。必要なセクションと前提変数を選んで実行してください。
Llama/GemmaのPPL・LoRAはCUDA GPUを想定。Gemma原本にはモデル削除後の参照、Llama原本には未定義変数の参照があります。
元の出力はDrive原本に保存。現在のコードで同じ結果を再現できることは未検証です。
保存先・Colab接続・既知の制約は `docs/notebook_execution.md` を参照。


In [ ]:
from pathlib import Path
import os, sys
# On a remote Colab kernel, first upload/clone this repository into /content/RMT_utils.
candidates = [Path.cwd(), *Path.cwd().parents, Path('/content/RMT_utils')]
ROOT = next((p for p in candidates if (p / 'notebook_runtime.py').is_file()), None)
if ROOT is None:
    raise RuntimeError('Upload or clone RMT_utils on this kernel, then rerun this cell.')
sys.path.insert(0, str(ROOT))
from notebook_runtime import setup_auth, data_dir, output_dir
setup_auth()
os.environ.setdefault('WANDB_MODE', 'offline')
print('Repository:', ROOT)
print('Data:', data_dir())
print('Outputs:', output_dir())


https://huggingface.co/google/gemma-3-4b-pt

In [ ]:
# Environment setup is managed outside this experiment cell.

import sys
sys.path.insert(0, str(ROOT))

# Environment setup is managed outside this experiment cell.
import funcs1
import evaluater

from evaluater import ppl_eval


# Colab-specific setup is handled by notebook_runtime.
import os

setup_auth()
# os.environ["WANDB_API_KEY"] = userdata.get("WANDB_API_KEY")

# import wandb
# wandb.login()



In [ ]:
import os
import torch
import numpy as np
import matplotlib.pyplot as plt
from transformers import AutoModelForCausalLM
# Colab-specific setup is handled by notebook_runtime.
# 1. シークレットからトークンを読み込んで環境変数にセット
setup_auth()

# 本家 Llama-3.2-3B のモデルIDを指定
model_id = "google/gemma-3-4b-pt"

print(f"{model_id} を読み込んでいます...")
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch.float16,
    low_cpu_mem_usage=True,
    device_map="cpu" # 念のため一度CPUに展開
)

# 言語モデル部分（Transformer層）のみを渡して実行
# ※お使いのモデルの読み込み形式に合わせてどちらかを選択してください
if hasattr(model, "language_model"):
    lm_model = model.language_model
elif hasattr(model.model, "language_model"):
    lm_model = model.model.language_model
else:
    lm_model = model.model

In [ ]:
del model, lm_model

In [ ]:
import pandas as pd
# Colab-specific setup is handled by notebook_runtime.
import os
import torch
import numpy as np
import matplotlib.pyplot as plt
from transformers import AutoModelForCausalLM
# Colab-specific setup is handled by notebook_runtime.
# 1. Google Drive のマウント
# Mount Drive explicitly using the Colab extension if needed.

# 保存先のディレクトリを作成（ご自身の環境に合わせて変更してください）
save_dir = str(data_dir())
os.makedirs(save_dir, exist_ok=True)

filename = "Gemma3-4B_esd_metrics.pkl"
filepath = os.path.join(save_dir, filename)

if os.path.exists(filepath):
    results = pd.read_pickle(filepath)
    print(f"✅ データの読み込みが完了しました: {filepath}")
    print(f"行数（層の数）: {len(results)}")
else:
    print(f"❌ ファイルが見つかりません: {filepath}")

# ESD

In [ ]:
# 言語モデル部分（Transformer層）のみを渡して実行
# ※お使いのモデルの読み込み形式に合わせてどちらかを選択してください
if hasattr(model, "language_model"):
    lm_model = model.language_model
elif hasattr(model.model, "language_model"):
    lm_model = model.model.language_model
else:
    lm_model = model.model

results = funcs1.get_esd_metrics(lm_model, pl_fitting='fix-finger')


import pandas as pd
import wandb
import pickle
import os
# Colab-specific setup is handled by notebook_runtime.
# 1. Google Drive のマウント
# Mount Drive explicitly using the Colab extension if needed.

# 保存先のディレクトリを作成（ご自身の環境に合わせて変更してください）
save_dir = str(data_dir())
os.makedirs(save_dir, exist_ok=True)

def save_esd_results(results, model_name="Gemma3-4B"):
    """
    results を Drive と WandB の両方に最適に保存する関数
    """
    # 辞書を Pandas DataFrame に変換
    df = pd.DataFrame(results)

    # ---------------------------------------------------------
    # ① Google Drive に完全な生データ（固有値含む）を保存 (Pickle形式)
    # ---------------------------------------------------------
    filename = f"{model_name}_esd_metrics.pkl"
    filepath = os.path.join(save_dir, filename)

    df.to_pickle(filepath)
    print(f"[Drive] 完全な結果を保存しました: {filepath}")

    # ---------------------------------------------------------
    # ② WandB に保存 (Artifacts と Table)
    # ---------------------------------------------------------
    # wandb の run が初期化されていない場合は初期化する
    if wandb.run is None:
        wandb.init(project="LlaMa-RMT-Analysis", name=f"ESD-Metrics-{model_name}")

    # 1. 完全な生データ(.pkl)を Artifact として WandB にバックアップ
    # これにより、後日別の環境からでも `wandb.use_artifact` で重い配列データを復元できます
    artifact = wandb.Artifact(name=f"{model_name}-esd-data", type="dataset")
    artifact.add_file(filepath)
    wandb.log_artifact(artifact)
    print(f"[WandB] 完全な生データを Artifact として保存しました。")

    # 2. WandB ダッシュボードで閲覧するための Table の作成
    # ブラウザの描画負荷を下げるため、数千個の要素を持つ 'eigs' 配列列のみを除外
    if 'eigs' in df.columns:
        df_for_table = df.drop(columns=['eigs'])
    else:
        df_for_table = df.copy()

    wandb_table = wandb.Table(dataframe=df_for_table)
    wandb.log({f"{model_name}_ESD_Metrics": wandb_table})
    print(f"[WandB] メトリクス一覧を Table として保存しました。ダッシュボードで確認できます。")

    # 最後に run を終了する（必要に応じてコメントアウトしてください）
    wandb.finish()


# 実行例
# results = get_esd_metrics(model)
save_esd_results(results, model_name="Gemma3-4B")


In [ ]:

import pandas as pd
import wandb
import pickle
import os
# Colab-specific setup is handled by notebook_runtime.
# 1. Google Drive のマウント
# Mount Drive explicitly using the Colab extension if needed.

# 保存先のディレクトリを作成（ご自身の環境に合わせて変更してください）
save_dir = str(data_dir())
os.makedirs(save_dir, exist_ok=True)

def save_esd_results(results, model_name="Gemma3-4B"):
    """
    results を Drive と WandB の両方に最適に保存する関数
    """
    # 辞書を Pandas DataFrame に変換
    df = pd.DataFrame(results)

    # ---------------------------------------------------------
    # ① Google Drive に完全な生データ（固有値含む）を保存 (Pickle形式)
    # ---------------------------------------------------------
    filename = f"{model_name}_esd_metrics.pkl"
    filepath = os.path.join(save_dir, filename)

    df.to_pickle(filepath)
    print(f"[Drive] 完全な結果を保存しました: {filepath}")

    # ---------------------------------------------------------
    # ② WandB に保存 (Artifacts と Table)
    # ---------------------------------------------------------
    # wandb の run が初期化されていない場合は初期化する
    if wandb.run is None:
        wandb.init(project="LlaMa-RMT-Analysis", name=f"ESD-Metrics-{model_name}")

    # 1. 完全な生データ(.pkl)を Artifact として WandB にバックアップ
    # これにより、後日別の環境からでも `wandb.use_artifact` で重い配列データを復元できます
    artifact = wandb.Artifact(name=f"{model_name}-esd-data", type="dataset")
    artifact.add_file(filepath)
    wandb.log_artifact(artifact)
    print(f"[WandB] 完全な生データを Artifact として保存しました。")

    # 2. WandB ダッシュボードで閲覧するための Table の作成
    # ブラウザの描画負荷を下げるため、数千個の要素を持つ 'eigs' 配列列のみを除外
    if 'eigs' in df.columns:
        df_for_table = df.drop(columns=['eigs'])
    else:
        df_for_table = df.copy()

    wandb_table = wandb.Table(dataframe=df_for_table)
    wandb.log({f"{model_name}_ESD_Metrics": wandb_table})
    print(f"[WandB] メトリクス一覧を Table として保存しました。ダッシュボードで確認できます。")

    # 最後に run を終了する（必要に応じてコメントアウトしてください）
    wandb.finish()


# 実行例
# results = get_esd_metrics(model)
save_esd_results(results, model_name="Gemma3-4B")


In [ ]:
results_numeric = results.select_dtypes(include=np.number)
display(results_numeric.describe())

In [ ]:
results_attn = results[results['name'].str.contains('attn')]
results_attn.describe()

In [ ]:
results_mlp = results[results['name'].str.contains('mlp')]
results_mlp.describe()

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

plt.figure(figsize=(14, 10))
corr_matrix = results_numeric.corr()

# -1から1のスケールで色分けするヒートマップ
sns.heatmap(corr_matrix,
            annot=True,       # セル内に数値を表示
            fmt=".2f",        # 小数点第2位まで表示
            cmap="coolwarm",  # 青から赤へのカラーマップ
            vmin=-1,          # 最小値
            vmax=1,           # 最大値
            linewidths=.5)    # セル間の境界線を設定

plt.title("Correlation Heatmap of results_numeric")
plt.tight_layout()
plt.show()

In [ ]:
plt.scatter(results_attn['alpha'], results_attn['s_hat_ratio_postDE'], label='Attention', color='blue', alpha=0.7)
plt.scatter(results_mlp['alpha'], results_mlp['s_hat_ratio_postDE'], label='MLP', color='orange', alpha=0.7)
plt.xlabel('alpha')
plt.ylabel('s_hat_ratio_postDE')
plt.legend()
plt.show()

In [ ]:
plt.scatter(results_attn['alpha'], results_attn['sigma2_postDE'], label='Attention', color='blue', alpha=0.7)
plt.scatter(results_mlp['alpha'], results_mlp['sigma2_postDE'], label='MLP', color='orange', alpha=0.7)
plt.xlabel('alpha')
plt.ylabel('sigma2_postDE')
plt.plot([1, 2.1], [1, 1], color='red')
plt.legend()
plt.show()

In [ ]:
# Attention層をさらに分割
results_q = results_attn[results_attn['name'].str.contains('q_proj')]
results_k = results_attn[results_attn['name'].str.contains('k_proj')]
results_v = results_attn[results_attn['name'].str.contains('v_proj')]
results_o = results_attn[results_attn['name'].str.contains('o_proj')]

# MLP層をさらに分割
results_gate = results_mlp[results_mlp['name'].str.contains('gate_proj')]
results_up = results_mlp[results_mlp['name'].str.contains('up_proj')]
results_down = results_mlp[results_mlp['name'].str.contains('down_proj')]

print(f"q: {len(results_q)}, k: {len(results_k)}, v: {len(results_v)}, o: {len(results_o)}")
print(f"gate: {len(results_gate)}, up: {len(results_up)}, down: {len(results_down)}")

In [ ]:
# Attention
plt.scatter(results_q['alpha'], results_q['s_hat_ratio_postDE'], label='q_proj', alpha=0.7)
plt.scatter(results_k['alpha'], results_k['s_hat_ratio_postDE'], label='k_proj', alpha=0.7)
plt.scatter(results_v['alpha'], results_v['s_hat_ratio_postDE'], label='v_proj', alpha=0.7)
plt.scatter(results_o['alpha'], results_o['s_hat_ratio_postDE'], label='o_proj', alpha=0.7)

# MLP
plt.scatter(results_gate['alpha'], results_gate['s_hat_ratio_postDE'], label='gate_proj', alpha=0.7)
plt.scatter(results_up['alpha'], results_up['s_hat_ratio_postDE'], label='up_proj', alpha=0.7)
plt.scatter(results_down['alpha'], results_down['s_hat_ratio_postDE'], label='down_proj', alpha=0.7)

plt.xlabel('alpha')
plt.ylabel('s_hat_ratio_postDE')

plt.legend()
plt.show()

In [ ]:
# Attention
plt.scatter(results_q['alpha'], results_q['sigma2_postDE'], label='q_proj', alpha=0.7)
plt.scatter(results_k['alpha'], results_k['sigma2_postDE'], label='k_proj', alpha=0.7)
plt.scatter(results_v['alpha'], results_v['sigma2_postDE'], label='v_proj', alpha=0.7)
plt.scatter(results_o['alpha'], results_o['sigma2_postDE'], label='o_proj', alpha=0.7)

# MLP
plt.scatter(results_gate['alpha'], results_gate['sigma2_postDE'], label='gate_proj', alpha=0.7)
plt.scatter(results_up['alpha'], results_up['sigma2_postDE'], label='up_proj', alpha=0.7)
plt.scatter(results_down['alpha'], results_down['sigma2_postDE'], label='down_proj', alpha=0.7)

plt.plot([1, 2.1], [1, 1], color='red')

plt.xlabel('alpha')
plt.ylabel('sigma2_postDE')

plt.legend()
plt.show()

In [ ]:
plt.hist(results_down['alpha'],bins=10)

In [ ]:
plt.figure(figsize=(10, 6))

# Attention
plt.plot(results_q['alpha'].values, label='q_proj', marker='o')
plt.plot(results_k['alpha'].values, label='k_proj', marker='o')
plt.plot(results_v['alpha'].values, label='v_proj', marker='o')
plt.plot(results_o['alpha'].values, label='o_proj', marker='o')

# MLP
plt.plot(results_gate['alpha'].values, label='gate_proj', marker='x')
plt.plot(results_up['alpha'].values, label='up_proj', marker='x')
plt.plot(results_down['alpha'].values, label='down_proj', marker='x')

plt.xlabel('Layer Index')
plt.ylabel('alpha')
plt.title('Alpha values across layers for 7 projections')

# 凡例を外側に配置
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.show()

# LRA

In [ ]:
import wandb
import random
import matplotlib.pyplot as plt
import torch
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer
from tqdm import tqdm

model_id = "google/gemma-3-4b-pt"

tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch.bfloat16,
    low_cpu_mem_usage=True,
    trust_remote_code=True
).to("cuda")

model.eval()

text = "This is a test sentence for checking model logits."
inputs = tokenizer(
    text,
    return_tensors="pt"
).to("cuda")
import wandb
import random
import matplotlib.pyplot as plt
import torch
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer
from tqdm import tqdm

# 事前に results, funcs1 などが定義・準備されている前提
# ※ results は lm_model を対象に get_esd_metrics で計算されたものとします

MAX_LAYERS = 20  # 例として最初の20層で比較
iter_count = 10
model_id = "google/gemma-3-4b-pt"

tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)
with torch.no_grad():
    outputs = model(
        **inputs,
        use_cache=False,
        return_dict=True
    )

logits = outputs.logits

print("logits dtype:", logits.dtype)
print("logits shape:", logits.shape)
print("all finite:", torch.isfinite(logits).all().item())
print("NaN count:", torch.isnan(logits).sum().item())
print("Inf count:", torch.isinf(logits).sum().item())

finite_logits = logits[torch.isfinite(logits)]

if finite_logits.numel() > 0:
    print("finite min:", finite_logits.min().item())
    print("finite max:", finite_logits.max().item())

In [ ]:
import wandb
import random
import matplotlib.pyplot as plt
import torch
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer
from tqdm import tqdm

# 事前に results, funcs1 などが定義・準備されている前提
# ※ results は lm_model を対象に get_esd_metrics で計算されたものとします

MAX_LAYERS = 20  # 例として最初の20層で比較
iter_count = 10
model_id = "google/gemma-3-4b-pt"

tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)

# --- WandB の初期化設定 ---
wandb_project_name = "LRA-Ablation-Study-Gemma3"

# ==========================================
# 実験: ランダム順での LRA (iter_count 回繰り返す)
# ==========================================
for i in range(iter_count):
    print(f"\n{'='*20}\n===== ランダム実験 {i+1}/{iter_count} ======\n{'='*20}")

    # 1. WandB の Run を初期化
    run_name = f"Random_Order_iter{i+1}"
    wandb.init(
        project=wandb_project_name,
        name=run_name,
        config={
            "method": "Random",
            "iteration": i + 1,
            "max_layers": MAX_LAYERS,
            "model_id": model_id,
            "DE": True
        },
        reinit=True
    )

    # 2. クリーンなモデルをロード (毎回初期化)
    print(f"{model_id} を読み込んでいます...")
    # model = AutoModelForCausalLM.from_pretrained(
    #     model_id,
    #     torch_dtype=torch.float16,
    #     low_cpu_mem_usage=True,
    #     device_map="cpu",
    #     trust_remote_code=True
    # )
    model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch.bfloat16,
    device_map="auto",
    low_cpu_mem_usage=True,
    trust_remote_code=True
    )

    # 💡 【重要な修正】
    # PPLの計算（logitsの出力）には一番外側の `model` が必要ですが、
    # `results` に記録されているレイヤー名にはプレフィックスがありません。
    # そこで、outer_model 内でのプレフィックスを特定し、自動付与します。
    if hasattr(model, "language_model"):
        lm_model = model.language_model
    elif hasattr(model, "model") and hasattr(model.model, "language_model"):
        lm_model = model.model.language_model
    elif hasattr(model, "model"):
        lm_model = model.model
    else:
        lm_model = model

    # 動的プレフィックスの取得 (例: "model." など)
    prefix = ""
    for name, mod in model.named_modules():
        if mod is lm_model:
            prefix = name + "." if name else ""
            break

    print(f"特定されたレイヤーのプレフィックス: '{prefix}'")

    # results のコピーを作成し、name列にプレフィックスを付ける
    results_outer = results.copy()
    results_outer['name'] = results_outer['name'].apply(lambda x: prefix + x if not x.startswith(prefix) else x)

    # Alphaが大きい順のリストを作成し、ランダムにシャッフル
    lra_list_alpha_outer = results_outer.sort_values(by='alpha', ascending=False)['name'].tolist()
    lra_list_random = lra_list_alpha_outer.copy()
    random.shuffle(lra_list_random)

    # 3. 実験を実行
    # ❗️ logitsを出力させるために、必ず一番外側の `model` を渡します
    history_random = funcs1.run_lra_experiment(
        model, tokenizer, results_outer,
        lra_list=lra_list_random,
        max_lra_layers=MAX_LAYERS,
        dataset_name='wikitext2',
        DE=True,
        seq_len=1024,
        batch_size=2
    )

    # 4. 結果を WandB にログとして送信
    for index, row in history_random.iterrows():
        wandb.log({
            "step": row['step'],
            "layer_name": row['layer_compressed'],
            "ppl_wikitext2": row['ppl'],
            "alpha_val": row['alpha_of_layer'],
            "reduction_ratio_percent": row['reduction_ratio_percent']
        })

    # 5. Matplotlib でもローカルに描画
    plt.plot(history_random['reduction_ratio_percent'], history_random['ppl'], alpha=0.5, label=f'Random {i+1}')

    # ==========================================
    # 🚨 OOMを防ぐためのメモリ完全解放処理
    # ==========================================
    del lm_model
    del model

    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()

    print(f"🧹 イテレーション {i+1} 終了: GPUメモリを解放しました。")
    wandb.finish()

# Matplotlib の仕上げ
plt.xlabel("Parameter Reduction Ratio (%)")
plt.ylabel("Wikitext-2 PPL")
plt.title(f"LRA Perplexity Degradation: Random Order ({model_id})")
plt.show()

In [ ]:
import wandb
import random
import matplotlib.pyplot as plt
import torch
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer
from tqdm import tqdm

# 事前に results, funcs1 などが定義・準備されている前提
# ※ results は lm_model を対象に get_esd_metrics で計算されたものとします

MAX_LAYERS = 20  # 例として最初の20層で比較
iter_count = 1
model_id = "google/gemma-3-4b-pt"

tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)

# --- WandB の初期化設定 ---
wandb_project_name = "LRA-Ablation-Study-Gemma3"

# ==========================================
# 実験: ランダム順での LRA (iter_count 回繰り返す)
# ==========================================
for i in range(iter_count):
    print(f"\n{'='*20}\n===== ランダム実験 {i+1}/{iter_count} ======\n{'='*20}")

    # 1. WandB の Run を初期化
    run_name = f"alpha_descending_iter{i+1}"
    wandb.init(
        project=wandb_project_name,
        name=run_name,
        config={
            "method": "alpha-descending",
            "iteration": i + 1,
            "max_layers": MAX_LAYERS,
            "model_id": model_id,
            "DE": True
        },
        reinit=True
    )

    # 2. クリーンなモデルをロード (毎回初期化)
    print(f"{model_id} を読み込んでいます...")
    # model = AutoModelForCausalLM.from_pretrained(
    #     model_id,
    #     torch_dtype=torch.float16,
    #     low_cpu_mem_usage=True,
    #     device_map="cpu",
    #     trust_remote_code=True
    # )
    model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch.bfloat16,
    device_map="auto",
    low_cpu_mem_usage=True,
    trust_remote_code=True
    )

    # 💡 【重要な修正】
    # PPLの計算（logitsの出力）には一番外側の `model` が必要ですが、
    # `results` に記録されているレイヤー名にはプレフィックスがありません。
    # そこで、outer_model 内でのプレフィックスを特定し、自動付与します。
    if hasattr(model, "language_model"):
        lm_model = model.language_model
    elif hasattr(model, "model") and hasattr(model.model, "language_model"):
        lm_model = model.model.language_model
    elif hasattr(model, "model"):
        lm_model = model.model
    else:
        lm_model = model

    # 動的プレフィックスの取得 (例: "model." など)
    prefix = ""
    for name, mod in model.named_modules():
        if mod is lm_model:
            prefix = name + "." if name else ""
            break

    print(f"特定されたレイヤーのプレフィックス: '{prefix}'")

    # results のコピーを作成し、name列にプレフィックスを付ける
    results_outer = results.copy()
    results_outer['name'] = results_outer['name'].apply(lambda x: prefix + x if not x.startswith(prefix) else x)

    # Alphaが大きい順のリストを作成し、ランダムにシャッフル
    lra_list_alpha_outer = results_outer.sort_values(by='alpha', ascending=False)['name'].tolist()

    # 3. 実験を実行
    # ❗️ logitsを出力させるために、必ず一番外側の `model` を渡します
    history_random = funcs1.run_lra_experiment(
        model, tokenizer, results_outer,
        lra_list=lra_list_alpha_outer,
        max_lra_layers=MAX_LAYERS,
        dataset_name='wikitext2',
        DE=True,
        seq_len=1024,
        batch_size=2
    )

    # 4. 結果を WandB にログとして送信
    for index, row in history_random.iterrows():
        wandb.log({
            "step": row['step'],
            "layer_name": row['layer_compressed'],
            "ppl_wikitext2": row['ppl'],
            "alpha_val": row['alpha_of_layer'],
            "reduction_ratio_percent": row['reduction_ratio_percent']
        })

    # 5. Matplotlib でもローカルに描画
    plt.plot(history_random['reduction_ratio_percent'], history_random['ppl'], alpha=0.5, label=f'Random {i+1}')

    # ==========================================
    # 🚨 OOMを防ぐためのメモリ完全解放処理
    # ==========================================
    del lm_model
    del model

    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()

    print(f"🧹 イテレーション {i+1} 終了: GPUメモリを解放しました。")
    wandb.finish()

# Matplotlib の仕上げ
plt.xlabel("Parameter Reduction Ratio (%)")
plt.ylabel("Wikitext-2 PPL")
plt.title(f"LRA Perplexity Degradation: alpha descending ({model_id})")
plt.show()

In [ ]:
import wandb
import random
import matplotlib.pyplot as plt
import torch
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer
from tqdm import tqdm

# 事前に results, funcs1 などが定義・準備されている前提
# ※ results は lm_model を対象に get_esd_metrics で計算されたものとします

MAX_LAYERS = 200  # 例として最初の20層で比較
iter_count = 1
model_id = "google/gemma-3-4b-pt"

tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)

# --- WandB の初期化設定 ---
wandb_project_name = "LRA-Ablation-Study-Gemma3"

# ==========================================
# 実験: ランダム順での LRA (iter_count 回繰り返す)
# ==========================================
for i in range(iter_count):
    print(f"\n{'='*20}\n===== ランダム実験 {i+1}/{iter_count} ======\n{'='*20}")

    # 1. WandB の Run を初期化
    run_name = f"alpha_ascending_iter{i+1}"
    wandb.init(
        project=wandb_project_name,
        name=run_name,
        config={
            "method": "alpha-ascending",
            "iteration": i + 1,
            "max_layers": MAX_LAYERS,
            "model_id": model_id,
            "DE": True
        },
        reinit=True
    )

    # 2. クリーンなモデルをロード (毎回初期化)
    print(f"{model_id} を読み込んでいます...")
    # model = AutoModelForCausalLM.from_pretrained(
    #     model_id,
    #     torch_dtype=torch.float16,
    #     low_cpu_mem_usage=True,
    #     device_map="cpu",
    #     trust_remote_code=True
    # )
    model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch.bfloat16,
    device_map="auto",
    low_cpu_mem_usage=True,
    trust_remote_code=True
    )

    # 💡 【重要な修正】
    # PPLの計算（logitsの出力）には一番外側の `model` が必要ですが、
    # `results` に記録されているレイヤー名にはプレフィックスがありません。
    # そこで、outer_model 内でのプレフィックスを特定し、自動付与します。
    if hasattr(model, "language_model"):
        lm_model = model.language_model
    elif hasattr(model, "model") and hasattr(model.model, "language_model"):
        lm_model = model.model.language_model
    elif hasattr(model, "model"):
        lm_model = model.model
    else:
        lm_model = model

    # 動的プレフィックスの取得 (例: "model." など)
    prefix = ""
    for name, mod in model.named_modules():
        if mod is lm_model:
            prefix = name + "." if name else ""
            break

    print(f"特定されたレイヤーのプレフィックス: '{prefix}'")

    # results のコピーを作成し、name列にプレフィックスを付ける
    results_outer = results.copy()
    results_outer['name'] = results_outer['name'].apply(lambda x: prefix + x if not x.startswith(prefix) else x)

    # Alphaが大きい順のリストを作成し、ランダムにシャッフル
    lra_list_alpha_outer = results_outer.sort_values(by='alpha', ascending=True)['name'].tolist()

    # 3. 実験を実行
    # ❗️ logitsを出力させるために、必ず一番外側の `model` を渡します
    history_random = funcs1.run_lra_experiment(
        model, tokenizer, results_outer,
        lra_list=lra_list_alpha_outer,
        max_lra_layers=MAX_LAYERS,
        dataset_name='wikitext2',
        DE=True,
        seq_len=1024,
        batch_size=2
    )

    # 4. 結果を WandB にログとして送信
    for index, row in history_random.iterrows():
        wandb.log({
            "step": row['step'],
            "layer_name": row['layer_compressed'],
            "ppl_wikitext2": row['ppl'],
            "alpha_val": row['alpha_of_layer'],
            "reduction_ratio_percent": row['reduction_ratio_percent']
        })

    # 5. Matplotlib でもローカルに描画
    plt.plot(history_random['reduction_ratio_percent'], history_random['ppl'], alpha=0.5, label=f'Random {i+1}')

    # ==========================================
    # 🚨 OOMを防ぐためのメモリ完全解放処理
    # ==========================================
    del lm_model
    del model

    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()

    print(f"🧹 イテレーション {i+1} 終了: GPUメモリを解放しました。")
    wandb.finish()

# Matplotlib の仕上げ
plt.xlabel("Parameter Reduction Ratio (%)")
plt.ylabel("Wikitext-2 PPL")
plt.title(f"LRA Perplexity Degradation: alpha ascending ({model_id})")
plt.show()

In [ ]:
import wandb
import random
import matplotlib.pyplot as plt
import torch
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer
from tqdm import tqdm

# 事前に results, funcs1 などが定義・準備されている前提
# ※ results は lm_model を対象に get_esd_metrics で計算されたものとします

MAX_LAYERS = 20  # 例として最初の20層で比較
iter_count = 1
model_id = "google/gemma-3-4b-pt"

tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)

# --- WandB の初期化設定 ---
wandb_project_name = "LRA-Ablation-Study-Gemma3"

# ==========================================
# 実験: ランダム順での LRA (iter_count 回繰り返す)
# ==========================================
for i in range(iter_count):
    print(f"\n{'='*20}\n===== ランダム実験 {i+1}/{iter_count} ======\n{'='*20}")

    # 1. WandB の Run を初期化
    run_name = f"KS_postDE_1_ascending_iter{i+1}"
    wandb.init(
        project=wandb_project_name,
        name=run_name,
        config={
            "method": "KS_postDE_1_ascending",
            "iteration": i + 1,
            "max_layers": MAX_LAYERS,
            "model_id": model_id,
            "DE": True
        },
        reinit=True
    )

    # 2. クリーンなモデルをロード (毎回初期化)
    print(f"{model_id} を読み込んでいます...")
    # model = AutoModelForCausalLM.from_pretrained(
    #     model_id,
    #     torch_dtype=torch.float16,
    #     low_cpu_mem_usage=True,
    #     device_map="cpu",
    #     trust_remote_code=True
    # )
    model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch.bfloat16,
    device_map="auto",
    low_cpu_mem_usage=True,
    trust_remote_code=True
    )

    # 💡 【重要な修正】
    # PPLの計算（logitsの出力）には一番外側の `model` が必要ですが、
    # `results` に記録されているレイヤー名にはプレフィックスがありません。
    # そこで、outer_model 内でのプレフィックスを特定し、自動付与します。
    if hasattr(model, "language_model"):
        lm_model = model.language_model
    elif hasattr(model, "model") and hasattr(model.model, "language_model"):
        lm_model = model.model.language_model
    elif hasattr(model, "model"):
        lm_model = model.model
    else:
        lm_model = model

    # 動的プレフィックスの取得 (例: "model." など)
    prefix = ""
    for name, mod in model.named_modules():
        if mod is lm_model:
            prefix = name + "." if name else ""
            break

    print(f"特定されたレイヤーのプレフィックス: '{prefix}'")

    # results のコピーを作成し、name列にプレフィックスを付ける
    results_outer = results.copy()
    results_outer['name'] = results_outer['name'].apply(lambda x: prefix + x if not x.startswith(prefix) else x)

    # Alphaが大きい順のリストを作成し、ランダムにシャッフル
    lra_list_alpha_outer = results_outer.sort_values(by='KS_postDE_1', ascending=True)['name'].tolist()

    # 3. 実験を実行
    # ❗️ logitsを出力させるために、必ず一番外側の `model` を渡します
    history_random = funcs1.run_lra_experiment(
        model, tokenizer, results_outer,
        lra_list=lra_list_alpha_outer,
        max_lra_layers=MAX_LAYERS,
        dataset_name='wikitext2',
        DE=True,
        seq_len=1024,
        batch_size=2
    )

    # 4. 結果を WandB にログとして送信
    for index, row in history_random.iterrows():
        wandb.log({
            "step": row['step'],
            "layer_name": row['layer_compressed'],
            "ppl_wikitext2": row['ppl'],
            "alpha_val": row['alpha_of_layer'],
            "reduction_ratio_percent": row['reduction_ratio_percent']
        })

    # 5. Matplotlib でもローカルに描画
    plt.plot(history_random['reduction_ratio_percent'], history_random['ppl'], alpha=0.5, label=f'Random {i+1}')

    # ==========================================
    # 🚨 OOMを防ぐためのメモリ完全解放処理
    # ==========================================
    del lm_model
    del model

    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()

    print(f"🧹 イテレーション {i+1} 終了: GPUメモリを解放しました。")
    wandb.finish()

# Matplotlib の仕上げ
plt.xlabel("Parameter Reduction Ratio (%)")
plt.ylabel("Wikitext-2 PPL")
plt.title(f"LRA Perplexity Degradation: KS_postDE_1aescending ({model_id})")
plt.show()

In [ ]:
import wandb
import random
import matplotlib.pyplot as plt
import torch
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer
from tqdm import tqdm

# 事前に results, funcs1 などが定義・準備されている前提
# ※ results は lm_model を対象に get_esd_metrics で計算されたものとします

MAX_LAYERS = 200  # 例として最初の20層で比較
iter_count = 1
model_id = "google/gemma-3-4b-pt"

tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)

# --- WandB の初期化設定 ---
wandb_project_name = "LRA-Ablation-Study-Gemma3"

# ==========================================
# 実験: ランダム順での LRA (iter_count 回繰り返す)
# ==========================================
for i in range(iter_count):
    print(f"\n{'='*20}\n===== ランダム実験 {i+1}/{iter_count} ======\n{'='*20}")

    # 1. WandB の Run を初期化
    run_name = f"KS_postDE_1_descending_iter{i+1}"
    wandb.init(
        project=wandb_project_name,
        name=run_name,
        config={
            "method": "KS_postDE_1_descending",
            "iteration": i + 1,
            "max_layers": MAX_LAYERS,
            "model_id": model_id,
            "DE": True
        },
        reinit=True
    )

    # 2. クリーンなモデルをロード (毎回初期化)
    print(f"{model_id} を読み込んでいます...")
    # model = AutoModelForCausalLM.from_pretrained(
    #     model_id,
    #     torch_dtype=torch.float16,
    #     low_cpu_mem_usage=True,
    #     device_map="cpu",
    #     trust_remote_code=True
    # )
    model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch.bfloat16,
    device_map="auto",
    low_cpu_mem_usage=True,
    trust_remote_code=True
    )

    # 💡 【重要な修正】
    # PPLの計算（logitsの出力）には一番外側の `model` が必要ですが、
    # `results` に記録されているレイヤー名にはプレフィックスがありません。
    # そこで、outer_model 内でのプレフィックスを特定し、自動付与します。
    if hasattr(model, "language_model"):
        lm_model = model.language_model
    elif hasattr(model, "model") and hasattr(model.model, "language_model"):
        lm_model = model.model.language_model
    elif hasattr(model, "model"):
        lm_model = model.model
    else:
        lm_model = model

    # 動的プレフィックスの取得 (例: "model." など)
    prefix = ""
    for name, mod in model.named_modules():
        if mod is lm_model:
            prefix = name + "." if name else ""
            break

    print(f"特定されたレイヤーのプレフィックス: '{prefix}'")

    # results のコピーを作成し、name列にプレフィックスを付ける
    results_outer = results.copy()
    results_outer['name'] = results_outer['name'].apply(lambda x: prefix + x if not x.startswith(prefix) else x)

    # Alphaが大きい順のリストを作成し、ランダムにシャッフル
    lra_list_alpha_outer = results_outer.sort_values(by='KS_postDE_1', ascending=False)['name'].tolist()

    # 3. 実験を実行
    # ❗️ logitsを出力させるために、必ず一番外側の `model` を渡します
    history_random = funcs1.run_lra_experiment(
        model, tokenizer, results_outer,
        lra_list=lra_list_alpha_outer,
        max_lra_layers=MAX_LAYERS,
        dataset_name='wikitext2',
        DE=True,
        seq_len=1024,
        batch_size=2
    )

    # 4. 結果を WandB にログとして送信
    for index, row in history_random.iterrows():
        wandb.log({
            "step": row['step'],
            "layer_name": row['layer_compressed'],
            "ppl_wikitext2": row['ppl'],
            "alpha_val": row['alpha_of_layer'],
            "reduction_ratio_percent": row['reduction_ratio_percent']
        })

    # 5. Matplotlib でもローカルに描画
    plt.plot(history_random['reduction_ratio_percent'], history_random['ppl'], alpha=0.5, label=f'Random {i+1}')

    # ==========================================
    # 🚨 OOMを防ぐためのメモリ完全解放処理
    # ==========================================
    del lm_model
    del model

    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()

    print(f"🧹 イテレーション {i+1} 終了: GPUメモリを解放しました。")
    wandb.finish()

# Matplotlib の仕上げ
plt.xlabel("Parameter Reduction Ratio (%)")
plt.ylabel("Wikitext-2 PPL")
plt.title(f"LRA Perplexity Degradation: KS_postDE_1_descending ({model_id})")
plt.show()

In [ ]:
import wandb
import random
import matplotlib.pyplot as plt
import torch
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer
from tqdm import tqdm

# 事前に results, funcs1 などが定義・準備されている前提
# ※ results は lm_model を対象に get_esd_metrics で計算されたものとします

MAX_LAYERS = 20  # 例として最初の20層で比較
iter_count = 1
model_id = "google/gemma-3-4b-pt"

tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)

# --- WandB の初期化設定 ---
wandb_project_name = "LRA-Ablation-Study-Gemma3"

# ==========================================
# 実験: ランダム順での LRA (iter_count 回繰り返す)
# ==========================================
for i in range(iter_count):
    print(f"\n{'='*20}\n===== ランダム実験 {i+1}/{iter_count} ======\n{'='*20}")

    # 1. WandB の Run を初期化
    run_name = f"s_hat_ratio_postDE_ascending_iter{i+1}"
    wandb.init(
        project=wandb_project_name,
        name=run_name,
        config={
            "method": "s_hat_ratio_postDE_ascending",
            "iteration": i + 1,
            "max_layers": MAX_LAYERS,
            "model_id": model_id,
            "DE": True
        },
        reinit=True
    )

    # 2. クリーンなモデルをロード (毎回初期化)
    print(f"{model_id} を読み込んでいます...")
    # model = AutoModelForCausalLM.from_pretrained(
    #     model_id,
    #     torch_dtype=torch.float16,
    #     low_cpu_mem_usage=True,
    #     device_map="cpu",
    #     trust_remote_code=True
    # )
    model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch.bfloat16,
    device_map="auto",
    low_cpu_mem_usage=True,
    trust_remote_code=True
    )

    # 💡 【重要な修正】
    # PPLの計算（logitsの出力）には一番外側の `model` が必要ですが、
    # `results` に記録されているレイヤー名にはプレフィックスがありません。
    # そこで、outer_model 内でのプレフィックスを特定し、自動付与します。
    if hasattr(model, "language_model"):
        lm_model = model.language_model
    elif hasattr(model, "model") and hasattr(model.model, "language_model"):
        lm_model = model.model.language_model
    elif hasattr(model, "model"):
        lm_model = model.model
    else:
        lm_model = model

    # 動的プレフィックスの取得 (例: "model." など)
    prefix = ""
    for name, mod in model.named_modules():
        if mod is lm_model:
            prefix = name + "." if name else ""
            break

    print(f"特定されたレイヤーのプレフィックス: '{prefix}'")

    # results のコピーを作成し、name列にプレフィックスを付ける
    results_outer = results.copy()
    results_outer['name'] = results_outer['name'].apply(lambda x: prefix + x if not x.startswith(prefix) else x)

    # Alphaが大きい順のリストを作成し、ランダムにシャッフル
    lra_list_alpha_outer = results_outer.sort_values(by='s_hat_ratio_postDE', ascending=True)['name'].tolist()

    # 3. 実験を実行
    # ❗️ logitsを出力させるために、必ず一番外側の `model` を渡します
    history_random = funcs1.run_lra_experiment(
        model, tokenizer, results_outer,
        lra_list=lra_list_alpha_outer,
        max_lra_layers=MAX_LAYERS,
        dataset_name='wikitext2',
        DE=True,
        seq_len=1024,
        batch_size=2
    )

    # 4. 結果を WandB にログとして送信
    for index, row in history_random.iterrows():
        wandb.log({
            "step": row['step'],
            "layer_name": row['layer_compressed'],
            "ppl_wikitext2": row['ppl'],
            "alpha_val": row['alpha_of_layer'],
            "reduction_ratio_percent": row['reduction_ratio_percent']
        })

    # 5. Matplotlib でもローカルに描画
    plt.plot(history_random['reduction_ratio_percent'], history_random['ppl'], alpha=0.5, label=f'Random {i+1}')

    # ==========================================
    # 🚨 OOMを防ぐためのメモリ完全解放処理
    # ==========================================
    del lm_model
    del model

    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()

    print(f"🧹 イテレーション {i+1} 終了: GPUメモリを解放しました。")
    wandb.finish()

# Matplotlib の仕上げ
plt.xlabel("Parameter Reduction Ratio (%)")
plt.ylabel("Wikitext-2 PPL")
plt.title(f"LRA Perplexity Degradation: s_hat_ratio_postDE_ascending ({model_id})")
plt.show()

In [ ]:
import wandb
import random
import matplotlib.pyplot as plt
import torch
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer
from tqdm import tqdm

# 事前に results, funcs1 などが定義・準備されている前提
# ※ results は lm_model を対象に get_esd_metrics で計算されたものとします

MAX_LAYERS = 200  # 例として最初の20層で比較
iter_count = 1
model_id = "google/gemma-3-4b-pt"

tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)

# --- WandB の初期化設定 ---
wandb_project_name = "LRA-Ablation-Study-Gemma3"

# ==========================================
# 実験: ランダム順での LRA (iter_count 回繰り返す)
# ==========================================
for i in range(iter_count):
    print(f"\n{'='*20}\n===== ランダム実験 {i+1}/{iter_count} ======\n{'='*20}")

    # 1. WandB の Run を初期化
    run_name = f"s_hat_ratio_postDE_descending_iter{i+1}"
    wandb.init(
        project=wandb_project_name,
        name=run_name,
        config={
            "method": "s_hat_ratio_postDE_descending",
            "iteration": i + 1,
            "max_layers": MAX_LAYERS,
            "model_id": model_id,
            "DE": True
        },
        reinit=True
    )

    # 2. クリーンなモデルをロード (毎回初期化)
    print(f"{model_id} を読み込んでいます...")
    # model = AutoModelForCausalLM.from_pretrained(
    #     model_id,
    #     torch_dtype=torch.float16,
    #     low_cpu_mem_usage=True,
    #     device_map="cpu",
    #     trust_remote_code=True
    # )
    model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch.bfloat16,
    device_map="auto",
    low_cpu_mem_usage=True,
    trust_remote_code=True
    )

    # 💡 【重要な修正】
    # PPLの計算（logitsの出力）には一番外側の `model` が必要ですが、
    # `results` に記録されているレイヤー名にはプレフィックスがありません。
    # そこで、outer_model 内でのプレフィックスを特定し、自動付与します。
    if hasattr(model, "language_model"):
        lm_model = model.language_model
    elif hasattr(model, "model") and hasattr(model.model, "language_model"):
        lm_model = model.model.language_model
    elif hasattr(model, "model"):
        lm_model = model.model
    else:
        lm_model = model

    # 動的プレフィックスの取得 (例: "model." など)
    prefix = ""
    for name, mod in model.named_modules():
        if mod is lm_model:
            prefix = name + "." if name else ""
            break

    print(f"特定されたレイヤーのプレフィックス: '{prefix}'")

    # results のコピーを作成し、name列にプレフィックスを付ける
    results_outer = results.copy()
    results_outer['name'] = results_outer['name'].apply(lambda x: prefix + x if not x.startswith(prefix) else x)

    # Alphaが大きい順のリストを作成し、ランダムにシャッフル
    lra_list_alpha_outer = results_outer.sort_values(by='s_hat_ratio_postDE', ascending=False)['name'].tolist()

    # 3. 実験を実行
    # ❗️ logitsを出力させるために、必ず一番外側の `model` を渡します
    history_random = funcs1.run_lra_experiment(
        model, tokenizer, results_outer,
        lra_list=lra_list_alpha_outer,
        max_lra_layers=MAX_LAYERS,
        dataset_name='wikitext2',
        DE=True,
        seq_len=1024,
        batch_size=2
    )

    # 4. 結果を WandB にログとして送信
    for index, row in history_random.iterrows():
        wandb.log({
            "step": row['step'],
            "layer_name": row['layer_compressed'],
            "ppl_wikitext2": row['ppl'],
            "alpha_val": row['alpha_of_layer'],
            "reduction_ratio_percent": row['reduction_ratio_percent']
        })

    # 5. Matplotlib でもローカルに描画
    plt.plot(history_random['reduction_ratio_percent'], history_random['ppl'], alpha=0.5, label=f'Random {i+1}')

    # ==========================================
    # 🚨 OOMを防ぐためのメモリ完全解放処理
    # ==========================================
    del lm_model
    del model

    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()

    print(f"🧹 イテレーション {i+1} 終了: GPUメモリを解放しました。")
    wandb.finish()

# Matplotlib の仕上げ
plt.xlabel("Parameter Reduction Ratio (%)")
plt.ylabel("Wikitext-2 PPL")
plt.title(f"LRA Perplexity Degradation: s_hat_ratio_postDE_descending ({model_id})")
plt.show()

In [ ]:
import wandb
import random
import matplotlib.pyplot as plt
import torch
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer
from tqdm import tqdm

# 事前に results, funcs1 などが定義・準備されている前提
# ※ results は lm_model を対象に get_esd_metrics で計算されたものとします

MAX_LAYERS = 200  # 例として最初の20層で比較
iter_count = 1
model_id = "google/gemma-3-4b-pt"

tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)

# --- WandB の初期化設定 ---
wandb_project_name = "LRA-Ablation-Study-Gemma3"

# ==========================================
# 実験: ランダム順での LRA (iter_count 回繰り返す)
# ==========================================
for i in range(iter_count):
    print(f"\n{'='*20}\n===== ランダム実験 {i+1}/{iter_count} ======\n{'='*20}")

    # 1. WandB の Run を初期化
    run_name = f"alpha_ascending_OffDE_iter{i+1}"
    wandb.init(
        project=wandb_project_name,
        name=run_name,
        config={
            "method": "alpha-ascending",
            "iteration": i + 1,
            "max_layers": MAX_LAYERS,
            "model_id": model_id,
            "DE": False
        },
        reinit=True
    )

    # 2. クリーンなモデルをロード (毎回初期化)
    print(f"{model_id} を読み込んでいます...")
    # model = AutoModelForCausalLM.from_pretrained(
    #     model_id,
    #     torch_dtype=torch.float16,
    #     low_cpu_mem_usage=True,
    #     device_map="cpu",
    #     trust_remote_code=True
    # )
    model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch.bfloat16,
    device_map="auto",
    low_cpu_mem_usage=True,
    trust_remote_code=True
    )

    # 💡 【重要な修正】
    # PPLの計算（logitsの出力）には一番外側の `model` が必要ですが、
    # `results` に記録されているレイヤー名にはプレフィックスがありません。
    # そこで、outer_model 内でのプレフィックスを特定し、自動付与します。
    if hasattr(model, "language_model"):
        lm_model = model.language_model
    elif hasattr(model, "model") and hasattr(model.model, "language_model"):
        lm_model = model.model.language_model
    elif hasattr(model, "model"):
        lm_model = model.model
    else:
        lm_model = model

    # 動的プレフィックスの取得 (例: "model." など)
    prefix = ""
    for name, mod in model.named_modules():
        if mod is lm_model:
            prefix = name + "." if name else ""
            break

    print(f"特定されたレイヤーのプレフィックス: '{prefix}'")

    # results のコピーを作成し、name列にプレフィックスを付ける
    results_outer = results.copy()
    results_outer['name'] = results_outer['name'].apply(lambda x: prefix + x if not x.startswith(prefix) else x)

    # Alphaが大きい順のリストを作成し、ランダムにシャッフル
    lra_list_alpha_outer = results_outer.sort_values(by='alpha', ascending=True)['name'].tolist()

    # 3. 実験を実行
    # ❗️ logitsを出力させるために、必ず一番外側の `model` を渡します
    history_random = funcs1.run_lra_experiment(
        model, tokenizer, results_outer,
        lra_list=lra_list_alpha_outer,
        max_lra_layers=MAX_LAYERS,
        dataset_name='wikitext2',
        DE=False,
        seq_len=1024,
        batch_size=2
    )

    # 4. 結果を WandB にログとして送信
    for index, row in history_random.iterrows():
        wandb.log({
            "step": row['step'],
            "layer_name": row['layer_compressed'],
            "ppl_wikitext2": row['ppl'],
            "alpha_val": row['alpha_of_layer'],
            "reduction_ratio_percent": row['reduction_ratio_percent']
        })

    # 5. Matplotlib でもローカルに描画
    plt.plot(history_random['reduction_ratio_percent'], history_random['ppl'], alpha=0.5, label=f'Random {i+1}')

    # ==========================================
    # 🚨 OOMを防ぐためのメモリ完全解放処理
    # ==========================================
    del lm_model
    del model

    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()

    print(f"🧹 イテレーション {i+1} 終了: GPUメモリを解放しました。")
    wandb.finish()

# Matplotlib の仕上げ
plt.xlabel("Parameter Reduction Ratio (%)")
plt.ylabel("Wikitext-2 PPL")
plt.title(f"LRA Perplexity Degradation: alpha ascending ({model_id})")
plt.show()

In [ ]:
import wandb
import random
import matplotlib.pyplot as plt
import torch
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer
from tqdm import tqdm

# 事前に results, funcs1 などが定義・準備されている前提
# ※ results は lm_model を対象に get_esd_metrics で計算されたものとします

MAX_LAYERS = 200  # 例として最初の20層で比較
iter_count = 1
model_id = "google/gemma-3-4b-pt"

tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)

# --- WandB の初期化設定 ---
wandb_project_name = "LRA-Ablation-Study-Gemma3"

# ==========================================
# 実験: ランダム順での LRA (iter_count 回繰り返す)
# ==========================================
for i in range(iter_count):
    print(f"\n{'='*20}\n===== ランダム実験 {i+1}/{iter_count} ======\n{'='*20}")

    # 1. WandB の Run を初期化
    run_name = f"KS_postDE_1_descending_OffDE_iter{i+1}"
    wandb.init(
        project=wandb_project_name,
        name=run_name,
        config={
            "method": "KS_postDE_1_descending",
            "iteration": i + 1,
            "max_layers": MAX_LAYERS,
            "model_id": model_id,
            "DE": False
        },
        reinit=True
    )

    # 2. クリーンなモデルをロード (毎回初期化)
    print(f"{model_id} を読み込んでいます...")
    # model = AutoModelForCausalLM.from_pretrained(
    #     model_id,
    #     torch_dtype=torch.float16,
    #     low_cpu_mem_usage=True,
    #     device_map="cpu",
    #     trust_remote_code=True
    # )
    model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch.bfloat16,
    device_map="auto",
    low_cpu_mem_usage=True,
    trust_remote_code=True
    )

    # 💡 【重要な修正】
    # PPLの計算（logitsの出力）には一番外側の `model` が必要ですが、
    # `results` に記録されているレイヤー名にはプレフィックスがありません。
    # そこで、outer_model 内でのプレフィックスを特定し、自動付与します。
    if hasattr(model, "language_model"):
        lm_model = model.language_model
    elif hasattr(model, "model") and hasattr(model.model, "language_model"):
        lm_model = model.model.language_model
    elif hasattr(model, "model"):
        lm_model = model.model
    else:
        lm_model = model

    # 動的プレフィックスの取得 (例: "model." など)
    prefix = ""
    for name, mod in model.named_modules():
        if mod is lm_model:
            prefix = name + "." if name else ""
            break

    print(f"特定されたレイヤーのプレフィックス: '{prefix}'")

    # results のコピーを作成し、name列にプレフィックスを付ける
    results_outer = results.copy()
    results_outer['name'] = results_outer['name'].apply(lambda x: prefix + x if not x.startswith(prefix) else x)

    # Alphaが大きい順のリストを作成し、ランダムにシャッフル
    lra_list_alpha_outer = results_outer.sort_values(by='KS_postDE_1', ascending=False)['name'].tolist()

    # 3. 実験を実行
    # ❗️ logitsを出力させるために、必ず一番外側の `model` を渡します
    history_random = funcs1.run_lra_experiment(
        model, tokenizer, results_outer,
        lra_list=lra_list_alpha_outer,
        max_lra_layers=MAX_LAYERS,
        dataset_name='wikitext2',
        DE=False,
        seq_len=1024,
        batch_size=2
    )

    # 4. 結果を WandB にログとして送信
    for index, row in history_random.iterrows():
        wandb.log({
            "step": row['step'],
            "layer_name": row['layer_compressed'],
            "ppl_wikitext2": row['ppl'],
            "alpha_val": row['alpha_of_layer'],
            "reduction_ratio_percent": row['reduction_ratio_percent']
        })

    # 5. Matplotlib でもローカルに描画
    plt.plot(history_random['reduction_ratio_percent'], history_random['ppl'], alpha=0.5, label=f'Random {i+1}')

    # ==========================================
    # 🚨 OOMを防ぐためのメモリ完全解放処理
    # ==========================================
    del lm_model
    del model

    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()

    print(f"🧹 イテレーション {i+1} 終了: GPUメモリを解放しました。")
    wandb.finish()

# Matplotlib の仕上げ
plt.xlabel("Parameter Reduction Ratio (%)")
plt.ylabel("Wikitext-2 PPL")
plt.title(f"LRA Perplexity Degradation: KS_postDE_1_descending ({model_id})")
plt.show()

In [ ]:
import wandb
import random
import matplotlib.pyplot as plt
import torch
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer
from tqdm import tqdm

# 事前に results, funcs1 などが定義・準備されている前提
# ※ results は lm_model を対象に get_esd_metrics で計算されたものとします

MAX_LAYERS = 200  # 例として最初の20層で比較
iter_count = 1
model_id = "google/gemma-3-4b-pt"

tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)

# --- WandB の初期化設定 ---
wandb_project_name = "LRA-Ablation-Study-Gemma3"

# ==========================================
# 実験: ランダム順での LRA (iter_count 回繰り返す)
# ==========================================
for i in range(iter_count):
    print(f"\n{'='*20}\n===== ランダム実験 {i+1}/{iter_count} ======\n{'='*20}")

    # 1. WandB の Run を初期化
    run_name = f"s_hat_ratio_postDE_descending_OffDE_iter{i+1}"
    wandb.init(
        project=wandb_project_name,
        name=run_name,
        config={
            "method": "s_hat_ratio_postDE_descending",
            "iteration": i + 1,
            "max_layers": MAX_LAYERS,
            "model_id": model_id,
            "DE": False
        },
        reinit=True
    )

    # 2. クリーンなモデルをロード (毎回初期化)
    print(f"{model_id} を読み込んでいます...")
    # model = AutoModelForCausalLM.from_pretrained(
    #     model_id,
    #     torch_dtype=torch.float16,
    #     low_cpu_mem_usage=True,
    #     device_map="cpu",
    #     trust_remote_code=True
    # )
    model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch.bfloat16,
    device_map="auto",
    low_cpu_mem_usage=True,
    trust_remote_code=True
    )

    # 💡 【重要な修正】
    # PPLの計算（logitsの出力）には一番外側の `model` が必要ですが、
    # `results` に記録されているレイヤー名にはプレフィックスがありません。
    # そこで、outer_model 内でのプレフィックスを特定し、自動付与します。
    if hasattr(model, "language_model"):
        lm_model = model.language_model
    elif hasattr(model, "model") and hasattr(model.model, "language_model"):
        lm_model = model.model.language_model
    elif hasattr(model, "model"):
        lm_model = model.model
    else:
        lm_model = model

    # 動的プレフィックスの取得 (例: "model." など)
    prefix = ""
    for name, mod in model.named_modules():
        if mod is lm_model:
            prefix = name + "." if name else ""
            break

    print(f"特定されたレイヤーのプレフィックス: '{prefix}'")

    # results のコピーを作成し、name列にプレフィックスを付ける
    results_outer = results.copy()
    results_outer['name'] = results_outer['name'].apply(lambda x: prefix + x if not x.startswith(prefix) else x)

    # Alphaが大きい順のリストを作成し、ランダムにシャッフル
    lra_list_alpha_outer = results_outer.sort_values(by='s_hat_ratio_postDE', ascending=False)['name'].tolist()

    # 3. 実験を実行
    # ❗️ logitsを出力させるために、必ず一番外側の `model` を渡します
    history_random = funcs1.run_lra_experiment(
        model, tokenizer, results_outer,
        lra_list=lra_list_alpha_outer,
        max_lra_layers=MAX_LAYERS,
        dataset_name='wikitext2',
        DE=False,
        seq_len=1024,
        batch_size=2
    )

    # 4. 結果を WandB にログとして送信
    for index, row in history_random.iterrows():
        wandb.log({
            "step": row['step'],
            "layer_name": row['layer_compressed'],
            "ppl_wikitext2": row['ppl'],
            "alpha_val": row['alpha_of_layer'],
            "reduction_ratio_percent": row['reduction_ratio_percent']
        })

    # 5. Matplotlib でもローカルに描画
    plt.plot(history_random['reduction_ratio_percent'], history_random['ppl'], alpha=0.5, label=f'Random {i+1}')

    # ==========================================
    # 🚨 OOMを防ぐためのメモリ完全解放処理
    # ==========================================
    del lm_model
    del model

    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()

    print(f"🧹 イテレーション {i+1} 終了: GPUメモリを解放しました。")
    wandb.finish()

# Matplotlib の仕上げ
plt.xlabel("Parameter Reduction Ratio (%)")
plt.ylabel("Wikitext-2 PPL")
plt.title(f"LRA Perplexity Degradation: s_hat_ratio_postDE_descending ({model_id})")
plt.show()

# LoRA

In [ ]:
# Environment setup is managed outside this experiment cell.
# Environment setup is managed outside this experiment cell.

In [ ]:
import importlib.util
import torch
import transformers
import peft
import accelerate

print("torch:", torch.__version__)
print("transformers:", transformers.__version__)
print("peft:", peft.__version__)
print("accelerate:", accelerate.__version__)
print("torchao installed:", importlib.util.find_spec("torchao") is not None)

In [ ]:
import os
import gc
import json
import random
import numpy as np
import pandas as pd
import torch

from datasets import load_dataset
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    DataCollatorForLanguageModeling,
    Trainer,
    TrainingArguments,
    set_seed,
)
from peft import (
    LoraConfig,
    TaskType,
    get_peft_model,
)

from funcs1 import get_ppl

In [ ]:
SEED = 42
set_seed(SEED)
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

model_id = "google/gemma-3-4b-pt"
MODEL_ID = "google/gemma-3-4b-pt"

# PPL評価条件
PPL_DATASET_NAME = "wikitext2"
PPL_SEQ_LEN = 1024
PPL_BATCH_SIZE = 2

# LoRA学習条件
TRAIN_SEQ_LEN = 1024
NUM_TRAIN_EPOCHS = 1
LEARNING_RATE = 2e-4

PER_DEVICE_TRAIN_BATCH_SIZE = 1
GRADIENT_ACCUMULATION_STEPS = 8

LORA_R = 8
LORA_ALPHA = 16
LORA_DROPOUT = 0.05

OUTPUT_ROOT = str(output_dir() / "lora_recovery_results")
os.makedirs(OUTPUT_ROOT, exist_ok=True)

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

# Llama tokenizerにはpad_tokenがない場合があるため設定
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

tokenizer.padding_side = "right"

print("pad_token:", tokenizer.pad_token)
print("pad_token_id:", tokenizer.pad_token_id)

In [ ]:
def prepare_wikitext2_train_dataset(
    tokenizer,
    seq_len=1024,
    num_proc=2,
):
    print("WikiText-2 train splitを読み込んでいます...")

    raw_train = load_dataset(
        "Salesforce/wikitext",
        "wikitext-2-raw-v1",
        split="train",
    )

    # 空行を除去
    raw_train = raw_train.filter(
        lambda example: example["text"] is not None
        and len(example["text"].strip()) > 0
    )

    def tokenize_function(examples):
        return tokenizer(
            examples["text"],
            add_special_tokens=False,
        )

    tokenized = raw_train.map(
        tokenize_function,
        batched=True,
        remove_columns=raw_train.column_names,
        num_proc=num_proc,
        desc="Tokenizing WikiText-2 train",
    )

    def group_texts(examples):
        # batched map内のtoken列を連結
        concatenated = {
            key: sum(examples[key], [])
            for key in examples.keys()
        }

        total_length = len(concatenated["input_ids"])

        # 端数を切り捨てる
        total_length = (total_length // seq_len) * seq_len

        result = {
            key: [
                values[i:i + seq_len]
                for i in range(0, total_length, seq_len)
            ]
            for key, values in concatenated.items()
        }

        return result

    lm_train_dataset = tokenized.map(
        group_texts,
        batched=True,
        num_proc=num_proc,
        desc=f"Grouping into {seq_len}-token blocks",
    )

    print("学習系列数:", len(lm_train_dataset))

    return lm_train_dataset

In [ ]:
train_dataset = prepare_wikitext2_train_dataset(
    tokenizer=tokenizer,
    seq_len=TRAIN_SEQ_LEN,
    num_proc=2,
)

print(train_dataset)
print("最初の系列長:", len(train_dataset[0]["input_ids"]))

In [ ]:
def load_fresh_model():
    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch.bfloat16,
    low_cpu_mem_usage=True,
    trust_remote_code=True
    ).to("cuda")

    model.config.use_cache = False

    return model

In [ ]:
def build_lra_model(
    strategy,
    max_lra_layers,
):
    """
    strategy:
      - "alpha_ascending"
      - "alpha_ascending_OffDE"
      - "KS_postDE_1_descending"
      - "KS_postDE_1_descending_OffDE"
    """

    model = load_fresh_model()
    if hasattr(model, "language_model"):
        lm_model = model.language_model
    elif hasattr(model, "model") and hasattr(model.model, "language_model"):
        lm_model = model.model.language_model
    elif hasattr(model, "model"):
        lm_model = model.model
    else:
        lm_model = model

    prefix = ""
    for name, mod in model.named_modules():
        if mod is lm_model:
            prefix = name + "." if name else ""
            break

    results_outer = results.copy()
    results_outer['name'] = results_outer['name'].apply(lambda x: prefix + x if not x.startswith(prefix) else x)


    if strategy == "alpha_ascending_DE":
        lra_list = (
            results_outer
            .sort_values(by="alpha", ascending=True)["name"]
            .tolist()
        )
        DE = True

    elif strategy == "alpha_ascending_OffDE":
        lra_list = (
            results_outer
            .sort_values(by="alpha", ascending=True)["name"]
            .tolist()
        )
        DE = False

    elif strategy == "KS_postDE_1_descending_DE":
        lra_list = (
            results_outer
            .sort_values(by="KS_postDE_1", ascending=False)["name"]
            .tolist()
        )
        DE = True

    elif strategy == "KS_postDE_1_descending_OffDE":
        lra_list = (
            results_outer
            .sort_values(by="KS_postDE_1", ascending=False)["name"]
            .tolist()
        )
        DE = False

    else:
        raise ValueError(f"Unknown LRA strategy: {strategy}")

    print("=" * 80)
    print("LRA strategy:", strategy)
    print("LRA layers:", max_lra_layers)
    print("=" * 80)

    # run_lra_experimentはmodelをin-placeで更新する前提
    history = funcs1.run_lra_experiment(
        model=model,
        tokenizer=tokenizer,
        results_df=results_outer,
        lra_list=lra_list,
        max_lra_layers=max_lra_layers,
        dataset_name=PPL_DATASET_NAME,
        DE=DE,
        PPLcalc=False, # 毎回のPPL計算をなくしてLRA後のモデルを一気に得る
        seq_len=PPL_SEQ_LEN,
        batch_size=PPL_BATCH_SIZE,
    )

    selected_layers = lra_list[:max_lra_layers]

    print("LRA完了")
    print("圧縮層数:", len(selected_layers))

    return model, history, selected_layers

In [ ]:
def build_pruned_model(
    strategy,
    target_sparsity,
):
    """
    strategy:
      - "uniform_magnitude"
      - "alpha_reverse_magnitude"
    """

    model = load_fresh_model()

    if strategy == "uniform_magnitude":
        alpha_prune = False
        alpha_reverse = False

    elif strategy == "alpha_reverse_magnitude":
        alpha_prune = True
        alpha_reverse = True

    else:
        raise ValueError(f"Unknown pruning strategy: {strategy}")

    print("=" * 80)
    print("Pruning strategy:", strategy)
    print("Target sparsity:", target_sparsity)
    print("=" * 80)

    # run_pruning_experimentはmodelの重みをin-placeで枝刈りする前提
    pruning_result = funcs1.run_pruning_experiment(
        model=model,
        tokenizer=tokenizer,
        results_df=results,
        dataset_name=PPL_DATASET_NAME,
        seq_len=PPL_SEQ_LEN,
        batch_size=PPL_BATCH_SIZE,
        sparsity=target_sparsity,
        alpha_prune=alpha_prune,
        prune_metric="magnitude",
        blockwise=False,
        alpha_reverse=alpha_reverse,
    )

    print("Pruning完了")
    print(pruning_result)

    return model, pruning_result

In [ ]:
def check_model_finite(model):
    bad_parameters = []

    for name, param in model.named_parameters():
        if not torch.is_floating_point(param):
            continue

        if not torch.isfinite(param).all():
            bad_parameters.append({
                "name": name,
                "nan": torch.isnan(param).sum().item(),
                "inf": torch.isinf(param).sum().item(),
                "shape": tuple(param.shape),
                "dtype": str(param.dtype),
            })

    if bad_parameters:
        print("❌ NaN/Infを含むパラメータがあります")
        for item in bad_parameters[:20]:
            print(item)
        raise RuntimeError("Model contains NaN/Inf")

    print("✅ 全パラメータはfiniteです")


In [ ]:
TARGET_MODULE_SUFFIXES = (
    "q_proj",
    "k_proj",
    "v_proj",
    "o_proj",
    "gate_proj",
    "up_proj",
    "down_proj",
)


def count_base_weight_sparsity(model):
    total = 0
    zeros = 0

    for name, module in model.named_modules():
        if not name.endswith(TARGET_MODULE_SUFFIXES):
            continue

        if not hasattr(module, "weight"):
            continue

        weight = module.weight.detach()

        total += weight.numel()
        zeros += (weight == 0).sum().item()

    sparsity = zeros / total if total > 0 else 0.0

    return {
        "zero_weights": zeros,
        "total_weights": total,
        "sparsity": sparsity,
    }

In [ ]:
LORA_TARGET_MODULES = [
    "q_proj",
    "k_proj",
    "v_proj",
    "o_proj",
    "gate_proj",
    "up_proj",
    "down_proj",
]


def attach_lora(model):
    model.config.use_cache = False

    lora_config = LoraConfig(
        task_type=TaskType.CAUSAL_LM,
        inference_mode=False,
        r=LORA_R,
        lora_alpha=LORA_ALPHA,
        lora_dropout=LORA_DROPOUT,
        bias="none",
        target_modules=LORA_TARGET_MODULES,
    )

    peft_model = get_peft_model(
        model,
        lora_config,
    )

    # メモリ削減
    peft_model.gradient_checkpointing_enable()
    peft_model.enable_input_require_grads()

    peft_model.print_trainable_parameters()

    return peft_model

In [ ]:
def evaluate_wikitext2_ppl(model):
    model.eval()

    ppl = get_ppl(
        model,
        tokenizer,
        dataset_name=PPL_DATASET_NAME,
        seq_len=PPL_SEQ_LEN,
        batch_size=PPL_BATCH_SIZE,
    )

    return float(ppl)

In [ ]:
def train_lora_adapter(
    model,
    experiment_name,
    train_dataset,
):
    output_dir = os.path.join(
        OUTPUT_ROOT,
        experiment_name,
    )

    os.makedirs(output_dir, exist_ok=True)

    data_collator = DataCollatorForLanguageModeling(
        tokenizer=tokenizer,
        mlm=False,
    )

    training_args = TrainingArguments(
        output_dir=output_dir,

        num_train_epochs=NUM_TRAIN_EPOCHS,
        learning_rate=LEARNING_RATE,

        per_device_train_batch_size=PER_DEVICE_TRAIN_BATCH_SIZE,
        gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,

        fp16=True,
        bf16=False,

        gradient_checkpointing=True,

        logging_strategy="steps",
        logging_steps=10,

        save_strategy="epoch",
        save_total_limit=1,

        report_to="none",

        optim="adamw_torch",
        weight_decay=0.0,
        # warmup_ratio=0.03,
        lr_scheduler_type="cosine",

        max_grad_norm=1.0,

        remove_unused_columns=False,
        dataloader_num_workers=2,

        seed=SEED,
        data_seed=SEED,
    )

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        data_collator=data_collator,
    )

    print("=" * 80)
    print("LoRA fine-tuning開始:", experiment_name)
    print("=" * 80)

    train_result = trainer.train()

    # Adapterのみ保存
    adapter_dir = os.path.join(
        output_dir,
        "final_adapter",
    )

    model.save_pretrained(adapter_dir)
    tokenizer.save_pretrained(adapter_dir)

    train_metrics = dict(train_result.metrics)

    with open(
        os.path.join(output_dir, "train_metrics.json"),
        "w",
        encoding="utf-8",
    ) as f:
        json.dump(
            train_metrics,
            f,
            ensure_ascii=False,
            indent=2,
        )

    return model, trainer, train_metrics

In [ ]:
def run_lora_recovery_experiment(
    experiment_name,
    compression_type,
    compression_strategy,
    train_dataset,
    max_lra_layers=None,
    target_sparsity=None,
):
    """
    compression_type:
      - "lra"
      - "pruning"
    """

    print("\n" + "#" * 100)
    print("Experiment:", experiment_name)
    print("#" * 100)

    # ---------------------------------------------------------
    # 1. 圧縮モデルをfresh modelから再構築
    # ---------------------------------------------------------
    compression_info = {}

    if compression_type == "lra":
        if max_lra_layers is None:
            raise ValueError("LRAではmax_lra_layersが必要です")

        model, lra_history, selected_layers = build_lra_model(
            strategy=compression_strategy,
            max_lra_layers=max_lra_layers,
        )

        compression_info = {
            "compression_type": "lra",
            "compression_strategy": compression_strategy,
            "num_lra_layers": max_lra_layers,
            "selected_layers": selected_layers,
        }

    elif compression_type == "pruning":
        if target_sparsity is None:
            raise ValueError("Pruningではtarget_sparsityが必要です")

        model, pruning_result = build_pruned_model(
            strategy=compression_strategy,
            target_sparsity=target_sparsity,
        )

        compression_info = {
            "compression_type": "pruning",
            "compression_strategy": compression_strategy,
            "target_sparsity": target_sparsity,
            "pruning_result": pruning_result,
        }

    else:
        raise ValueError(
            f"Unknown compression_type: {compression_type}"
        )

    # ---------------------------------------------------------
    # 2. 数値チェック
    # ---------------------------------------------------------
    check_model_finite(model)

    sparsity_before_lora = None

    if compression_type == "pruning":
        sparsity_before_lora = count_base_weight_sparsity(model)
        print("LoRA前のbase-weight sparsity:", sparsity_before_lora)

    # ---------------------------------------------------------
    # 3. LoRA前のPPL
    # ---------------------------------------------------------
    print("LoRA前PPLを測定します")

    ppl_before_lora = evaluate_wikitext2_ppl(model)

    print(
        f"✅ {experiment_name} "
        f"LoRA前 PPL = {ppl_before_lora:.4f}"
    )

    # ---------------------------------------------------------
    # 4. LoRA adapterを追加
    # ---------------------------------------------------------
    model = attach_lora(model)

    # ---------------------------------------------------------
    # 5. LoRA学習
    # ---------------------------------------------------------
    model, trainer, train_metrics = train_lora_adapter(
        model=model,
        experiment_name=experiment_name,
        train_dataset=train_dataset,
    )

    # ---------------------------------------------------------
    # 6. LoRA後のPPL
    # ---------------------------------------------------------
    print("LoRA後PPLを測定します")

    ppl_after_lora = evaluate_wikitext2_ppl(model)

    print(
        f"✅ {experiment_name} "
        f"LoRA後 PPL = {ppl_after_lora:.4f}"
    )

    # ---------------------------------------------------------
    # 7. Pruningモデルのゼロが維持されているか確認
    # ---------------------------------------------------------
    sparsity_after_lora = None

    if compression_type == "pruning":
        sparsity_after_lora = count_base_weight_sparsity(model)

        print(
            "LoRA後のbase-weight sparsity:",
            sparsity_after_lora,
        )

    # ---------------------------------------------------------
    # 8. 回復率
    # ---------------------------------------------------------
    baseline_ppl = 16.3933

    degradation_before = ppl_before_lora - baseline_ppl
    degradation_after = ppl_after_lora - baseline_ppl

    if degradation_before > 0:
        recovery_ratio = (
            degradation_before - degradation_after
        ) / degradation_before
    else:
        recovery_ratio = np.nan

    summary = {
        "experiment_name": experiment_name,
        "compression_type": compression_type,
        "compression_strategy": compression_strategy,

        "ppl_baseline": baseline_ppl,
        "ppl_before_lora": ppl_before_lora,
        "ppl_after_lora": ppl_after_lora,

        "absolute_ppl_improvement":
            ppl_before_lora - ppl_after_lora,

        "recovery_ratio":
            recovery_ratio,

        "lora_r": LORA_R,
        "lora_alpha": LORA_ALPHA,
        "lora_dropout": LORA_DROPOUT,
        "learning_rate": LEARNING_RATE,
        "num_train_epochs": NUM_TRAIN_EPOCHS,
        "train_seq_len": TRAIN_SEQ_LEN,

        "sparsity_before_lora":
            None if sparsity_before_lora is None
            else sparsity_before_lora["sparsity"],

        "sparsity_after_lora":
            None if sparsity_after_lora is None
            else sparsity_after_lora["sparsity"],
    }

    summary.update({
        key: value
        for key, value in compression_info.items()
        if key not in ["selected_layers", "pruning_result"]
    })

    experiment_dir = os.path.join(
        OUTPUT_ROOT,
        experiment_name,
    )

    with open(
        os.path.join(experiment_dir, "summary.json"),
        "w",
        encoding="utf-8",
    ) as f:
        json.dump(
            summary,
            f,
            ensure_ascii=False,
            indent=2,
            default=str,
        )

    print("\nSummary:")
    for key, value in summary.items():
        print(f"{key}: {value}")

    # ---------------------------------------------------------
    # 9. メモリ解放前にtrainer参照を削除
    # ---------------------------------------------------------
    del trainer

    return summary, model

In [ ]:
import torch
from transformers import AutoModelForCausalLM
from peft import LoraConfig, TaskType, get_peft_model


test_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.bfloat16,
    low_cpu_mem_usage=True,
    trust_remote_code=True
    ).to("cuda")

test_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    bias="none",
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    ],
)

test_model = get_peft_model(test_model, test_config)
test_model.print_trainable_parameters()

In [ ]:
EXPERIMENTS = [
    {
        "experiment_name":
            "lra_alpha_ascending_50layers",

        "compression_type":
            "lra",

        "compression_strategy":
            "alpha_ascending_DE",

        "max_lra_layers":
            50,
    },

    {
        "experiment_name":
            "lra_alpha_ascending_OffDE_50layers",

        "compression_type":
            "lra",

        "compression_strategy":
            "alpha_ascending_OffDE",

        "max_lra_layers":
            50,
    },
{
        "experiment_name":
            "lra_KS_postDE_1_descending_50layers",

        "compression_type":
            "lra",

        "compression_strategy":
            "KS_postDE_1_descending_DE",

        "max_lra_layers":
            50,
    },
    {
        "experiment_name":
            "lra_KS_postDE_1_descending_50layers",

        "compression_type":
            "lra",

        "compression_strategy":
            "KS_postDE_1_descending_OffDE",

        "max_lra_layers":
            50,
    },

]

In [ ]:
all_summaries = []

for exp in EXPERIMENTS:
    try:
        summary, trained_model = run_lora_recovery_experiment(
            experiment_name=exp["experiment_name"],
            compression_type=exp["compression_type"],
            compression_strategy=exp["compression_strategy"],
            train_dataset=train_dataset,
            max_lra_layers=exp.get("max_lra_layers"),
            target_sparsity=exp.get("target_sparsity"),
        )

        all_summaries.append(summary)

    except Exception as e:
        print(
            f"❌ Experiment failed: "
            f"{exp['experiment_name']}"
        )
        print(type(e).__name__, e)

        all_summaries.append({
            "experiment_name":
                exp["experiment_name"],
            "status":
                "failed",
            "error":
                repr(e),
        })

    finally:
        if "trained_model" in locals():
            del trained_model

        gc.collect()

        if torch.cuda.is_available():
            torch.cuda.empty_cache()
            torch.cuda.ipc_collect()

        # 途中までの結果も毎回保存
        pd.DataFrame(all_summaries).to_csv(
            os.path.join(
                OUTPUT_ROOT,
                "lora_recovery_summary.csv",
            ),
            index=False,
        )

        print("GPUメモリを解放しました")


In [ ]:
import pandas as pd
summary_df = pd.DataFrame(all_summaries)

display_columns = [
    "experiment_name",
    "compression_type",
    "compression_strategy",
    "ppl_baseline",
    "ppl_before_lora",
    "ppl_after_lora",
    "absolute_ppl_improvement",
    "recovery_ratio",
    "sparsity_before_lora",
    "sparsity_after_lora",
]

existing_columns = [
    col
    for col in display_columns
    if col in summary_df.columns
]

display(summary_df[existing_columns])


In [ ]:
# Colab-specific setup is handled by notebook_runtime.
# End the Colab server manually after saving your results.